## 1. Find post sitemaps

In [ ]:
# Find Kontrast post sitemaps in the sitemap index.
import re
import urllib.request
import xml.etree.ElementTree as ET
from urllib.parse import urlparse

index_url = "https://kontrast.at/sitemap_index.xml"
request = urllib.request.Request(index_url, headers={"User-Agent": "news-analytics-scraper/0.1"})
with urllib.request.urlopen(request, timeout=20) as response:
    root = ET.fromstring(response.read())

post_sitemaps = []
for loc in root.findall(".//{*}sitemap/{*}loc"):
    sitemap_url = loc.text.strip()
    filename = urlparse(sitemap_url).path.rsplit("/", 1)[-1]
    if re.fullmatch(r"post-sitemap(?:\d+)?\.xml", filename):
        post_sitemaps.append(sitemap_url)
        print(sitemap_url)

## 2. Inspect article URLs and lastmod

In [ ]:
# Preview article URLs and lastmod values from the first post sitemap.
request = urllib.request.Request(post_sitemaps[0], headers={"User-Agent": "news-analytics-scraper/0.1"})
with urllib.request.urlopen(request, timeout=20) as response:
    post_root = ET.fromstring(response.read())

print(f"Articles from {post_sitemaps[0]}:")
for article in post_root.findall("{*}url")[:5]:
    print(f"{article.findtext('{*}loc')} | lastmod={article.findtext('{*}lastmod')}")

## 3. Fetch one article as raw HTML

In [ ]:
# Fetch the first article as raw HTML bytes.
article_url = post_root.findtext("{*}url/{*}loc")
request = urllib.request.Request(article_url, headers={"User-Agent": "news-analytics-scraper/0.1"})
with urllib.request.urlopen(request, timeout=20) as response:
    html = response.read()
    print(f"Fetched {response.url}: HTTP {response.status}, {len(html)} bytes")
    print(f"Content-Type: {response.headers.get('Content-Type')}")
    print(html)